<a href="https://colab.research.google.com/github/abdullahh-sheikhh/voxelmorph/blob/dev/scripts/cell_tracking/train_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cell Deformation Estimation with VoxelMorph

2D deformable registration for PhC-C2DH-U373 cells using VoxelMorph, compared against Horn & Schunck classical optical flow baseline.

## 1. Setup

Clone the repo and install dependencies. Idempotent — safe to re-run.

In [ ]:
%cd /content
![ -d voxelmorph/.git ] || git clone https://github.com/abdullahh-sheikhh/voxelmorph.git
%cd /content/voxelmorph
!git checkout dev
!git pull
!pip install -e . -q
!pip install -r scripts/cell_tracking/requirements.txt -q
!pip install 'numpy<2.1' -q  # avoid scipy/skimage conflict on Colab

## 2. Imports

All Python imports for the notebook. Run after section 1.

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import ListedColormap
import torch
from skimage import io
from IPython.display import Image, display

from scripts.cell_tracking.cell_segmentation import SegmentedCellBuilder

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 3. Dataset Download and Setup

PhC-C2DH-U373 training data (sequences 01 and 02) from the Cell Tracking


In [ ]:
!mkdir -p dataset

if not Path('dataset/train/01').exists():
    print('Downloading training data...')
    !wget -q https://data.celltrackingchallenge.net/training-datasets/PhC-C2DH-U373.zip -O dataset/train.zip
    !cd dataset && unzip -q train.zip -d train_raw && mv train_raw/PhC-C2DH-U373/* train/ && rm -rf train_raw train.zip
else:
    print('Training data already exists')

!echo "Train sequences:" && ls dataset/train/01/ | head -3 && echo "... (115 frames total)"

## 4. Dataset Visualize

In [ ]:
#  Load two consecutive frames and their Silver Truth masks ─
frame_i, frame_j = 10, 11
seq_dir = Path('dataset/train/01')
st_dir  = Path('dataset/train/01_ST/SEG')

img_i  = io.imread(str(seq_dir / f't{frame_i:03d}.tif')).astype(np.float32) / 255.0
img_j  = io.imread(str(seq_dir / f't{frame_j:03d}.tif')).astype(np.float32) / 255.0
diff   = np.abs(img_j - img_i)

mask_i = io.imread(str(st_dir / f'man_seg{frame_i:03d}.tif')).astype(np.float32)
mask_j = io.imread(str(st_dir / f'man_seg{frame_j:03d}.tif')).astype(np.float32)

binary_i = mask_i > 0
binary_j = mask_j > 0
n_cells  = int(mask_i.max())

# Per-cell colour map: background = black, each cell = distinct colour
cell_cmap = ListedColormap(['black'] + [cm.tab10(i % 10) for i in range(n_cells)])

# Alignment overlay: red = source cells, green = target cells, yellow = overlap
align_rgb         = np.zeros((*img_i.shape, 3), dtype=np.float32)
align_rgb[..., 0] = binary_i.astype(float)
align_rgb[..., 1] = binary_j.astype(float)

#  Figure: 3 rows × 3 panels ─
fig, axes = plt.subplots(3, 3, figsize=(15, 12))

kw_img = dict(cmap='gray', vmin=0, vmax=1)
kw_src = dict(colors=['#00d4ff'], linewidths=0.9, levels=[0.5])   # cyan  = source
kw_tgt = dict(colors=['#ffcc00'], linewidths=0.9, levels=[0.5])   # yellow = target

#  Row 0: single frame anatomy ─
axes[0, 0].imshow(img_i, **kw_img)
axes[0, 0].set_title(
    f'Frame t={frame_i}  —  raw phase-contrast image\n'
    f'696×520 px  |  8-bit grayscale  |  0.65 μm/pixel  |  15 min/frame', fontsize=9)

axes[0, 1].imshow(img_i, **kw_img)
axes[0, 1].contour(binary_i.astype(float), **kw_src)
axes[0, 1].set_title(
    f'Silver Truth cell regions  ({n_cells} cells)\n'
    f'Cyan contours = ST masks  —  used as training supervision', fontsize=9)

axes[0, 2].imshow(mask_i, cmap=cell_cmap, vmin=0, vmax=n_cells, interpolation='nearest')
axes[0, 2].set_title(
    f'Cell label map  (0 = background, 1–{n_cells} = individual cells)\n'
    f'Integer IDs are stable across frames (tracking IDs)', fontsize=9)

#  Row 1: consecutive pair
axes[1, 0].imshow(img_i, **kw_img)
axes[1, 0].contour(binary_i.astype(float), **kw_src)
axes[1, 0].set_title(f'Source frame  t={frame_i}  (cyan = cell boundaries)', fontsize=9)

axes[1, 1].imshow(img_j, **kw_img)
axes[1, 1].contour(binary_j.astype(float), **kw_tgt)
axes[1, 1].set_title(f'Target frame  t={frame_j}  (+15 min)  (yellow = cell boundaries)', fontsize=9)

im = axes[1, 2].imshow(diff, cmap='inferno', vmin=0, vmax=diff.max())
axes[1, 2].set_title('|Target − Source|  —  where intensity changed\nBright = cells moved / deformed', fontsize=9)
fig.colorbar(im, ax=axes[1, 2], fraction=0.046, pad=0.04)

#  Row 2: the registration problem ─
axes[2, 0].imshow(binary_i.astype(float), cmap='gray', vmin=0, vmax=1)
axes[2, 0].set_title(
    f'Source mask  (binary)\n'
    f'Cell area = {binary_i.mean()*100:.1f}% of image pixels', fontsize=9)

axes[2, 1].imshow(binary_j.astype(float), cmap='gray', vmin=0, vmax=1)
axes[2, 1].set_title(
    f'Target mask  (binary)\n'
    f'Cell area = {binary_j.mean()*100:.1f}% of image pixels', fontsize=9)

axes[2, 2].imshow(align_rgb)
axes[2, 2].set_title(
    'Goal: warp source mask → target mask\n'
    'Red = source  |  Green = target  |  Yellow = overlap', fontsize=9)

#  Row labels & final formatting ─
row_labels = ['What the\ndata looks like', 'What we\nregister', 'The alignment\nproblem']
for row, label in enumerate(row_labels):
    axes[row, 0].annotate(
        label, xy=(-0.14, 0.5), xycoords='axes fraction',
        ha='center', va='center', fontsize=8, fontweight='bold',
        rotation=90, color='#444444')

for ax in axes.flat:
    ax.axis('off')

plt.suptitle(
    'PhC-C2DH-U373  —  Glioblastoma-astrocytoma U373 cells  |  Phase-contrast microscopy  |  Seq 01\n'
    '115 frames  ·  228 consecutive pairs  ·  3–8 cells per frame  ·  Silver Truth masks on all frames',
    fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 5. Trainings


### 5.1 Masked MSE

Train **Masked MSE** for 200 epochs on sequence 01.

Loss terms:
- `--mask-weight 1.0` — soft Dice on cell masks (primary alignment signal)
- `--int-weight 0.1` — cell-restricted intensity loss
- `--lambda 0.01` — displacement field smoothness regularisation

In [5]:
!python -m scripts.cell_tracking.train \
    --data-dir dataset/train \
    --sequences 01 \
    --epochs 200 --batch-size 4 --lr 1e-4 \
    --loss mse --lambda 0.01 \
    --mask-weight 1.0 --int-weight 0.1 \
    --output-dir output/masked_mse \
    --save-every 50

Device: cuda  |  Dataset: 114 pairs  |  Sequences: ['01']  |  Loss: MSE  |  Mode: mask-guided  |  lambda: 0.01

Epochs:   0% 0/200 [00:00<?, ?it/s]Epoch 1/200 — mask_dice_loss: 0.099433
Epochs:   4% 9/200 [00:44<15:18,  4.81s/it]Epoch 10/200 — mask_dice_loss: 0.097732
Epochs:  10% 19/200 [01:32<14:55,  4.95s/it]Epoch 20/200 — mask_dice_loss: 0.085674
Epochs:  14% 29/200 [02:21<13:57,  4.90s/it]Epoch 30/200 — mask_dice_loss: 0.081425
Epochs:  20% 39/200 [03:11<13:27,  5.02s/it]Epoch 40/200 — mask_dice_loss: 0.072052
Epochs:  24% 49/200 [04:00<12:30,  4.97s/it]Epoch 50/200 — mask_dice_loss: 0.066303
Epochs:  30% 59/200 [04:50<11:41,  4.97s/it]Epoch 60/200 — mask_dice_loss: 0.058781
Epochs:  34% 69/200 [05:39<10:53,  4.99s/it]Epoch 70/200 — mask_dice_loss: 0.054605
Epochs:  40% 79/200 [06:29<10:00,  4.96s/it]Epoch 80/200 — mask_dice_loss: 0.051720
Epochs:  44% 89/200 [07:18<09:12,  4.97s/it]Epoch 90/200 — mask_dice_loss: 0.048222
Epochs:  50% 99/200 [08:08<08:22,  4.97s/it]Epoch 100/200 —

In [ ]:
display(Image(filename='output/masked_mse/loss_curve.png', width=800))

### 5.2 NCC

Train **Normalized Cross-Correlation** for 200 epochs on sequence 01.

Loss terms:
- `--mask-weight 1.0` — soft Dice on cell masks
- `--int-weight 0.1` — cell-restricted intensity loss
- `--lambda 1.0` — displacement field smoothness regularisation

In [7]:
!python -m scripts.cell_tracking.train \
    --data-dir dataset/train \
    --sequences 01 \
    --epochs 200 --batch-size 4 --lr 1e-4 \
    --loss ncc --lambda 1.0 \
    --mask-weight 1.0 --int-weight 0.1 \
    --output-dir output/ncc \
    --save-every 50

Device: cuda  |  Dataset: 114 pairs  |  Sequences: ['01']  |  Loss: NCC  |  Mode: mask-guided  |  lambda: 1.0

Epochs:   0% 0/200 [00:00<?, ?it/s]Epoch 1/200 — mask_dice_loss: 0.099350
Epochs:   4% 9/200 [00:48<16:56,  5.32s/it]Epoch 10/200 — mask_dice_loss: 0.099159
Epochs:  10% 19/200 [01:40<15:36,  5.17s/it]Epoch 20/200 — mask_dice_loss: 0.087500
Epochs:  14% 29/200 [02:31<14:37,  5.13s/it]Epoch 30/200 — mask_dice_loss: 0.074045
Epochs:  20% 39/200 [03:23<13:43,  5.11s/it]Epoch 40/200 — mask_dice_loss: 0.069545
Epochs:  24% 49/200 [04:14<12:53,  5.12s/it]Epoch 50/200 — mask_dice_loss: 0.065257
Epochs:  30% 59/200 [05:06<12:10,  5.18s/it]Epoch 60/200 — mask_dice_loss: 0.063129
Epochs:  34% 69/200 [05:58<11:21,  5.20s/it]Epoch 70/200 — mask_dice_loss: 0.060444
Epochs:  40% 79/200 [06:49<10:20,  5.13s/it]Epoch 80/200 — mask_dice_loss: 0.058314
Epochs:  44% 89/200 [07:40<09:30,  5.14s/it]Epoch 90/200 — mask_dice_loss: 0.057253
Epochs:  50% 99/200 [08:31<08:34,  5.10s/it]Epoch 100/200 — 

In [ ]:
display(Image(filename='output/ncc/loss_curve.png', width=800))

### 5.3 SSIM

Train **SSIM** for 200 epochs on sequence 01.

Loss terms:
- `--mask-weight 1.0` — soft Dice on cell masks
- `--int-weight 0.1` — cell-restricted intensity loss
- `--lambda 1.0` — displacement field smoothness regularisation

In [ ]:
!python -m scripts.cell_tracking.train \
    --data-dir dataset/train \
    --sequences 01 \
    --epochs 200 --batch-size 4 --lr 1e-4 \
    --loss ssim --lambda 1.0 \
    --mask-weight 1.0 --int-weight 0.1 \
    --output-dir output/ssim \
    --save-every 50

Device: cuda  |  Dataset: 114 pairs  |  Sequences: ['01']  |  Loss: SSIM  |  Mode: mask-guided  |  lambda: 1.0

Epochs:   0% 0/200 [00:00<?, ?it/s]Epoch 1/200 — mask_dice_loss: 0.099624
Epochs:   4% 9/200 [00:46<16:13,  5.10s/it]Epoch 10/200 — mask_dice_loss: 0.096048
Epochs:  10% 19/200 [01:37<15:22,  5.10s/it]Epoch 20/200 — mask_dice_loss: 0.083159
Epochs:  14% 29/200 [02:28<14:36,  5.12s/it]Epoch 30/200 — mask_dice_loss: 0.073616
Epochs:  20% 39/200 [03:19<13:40,  5.10s/it]Epoch 40/200 — mask_dice_loss: 0.068374
Epochs:  24% 49/200 [04:11<12:56,  5.14s/it]Epoch 50/200 — mask_dice_loss: 0.064080
Epochs:  30% 59/200 [05:02<12:12,  5.19s/it]Epoch 60/200 — mask_dice_loss: 0.061510
Epochs:  34% 69/200 [05:54<11:17,  5.17s/it]Epoch 70/200 — mask_dice_loss: 0.060034
Epochs:  40% 79/200 [06:45<10:22,  5.15s/it]Epoch 80/200 — mask_dice_loss: 0.059499
Epochs:  44% 89/200 [07:37<09:31,  5.15s/it]Epoch 90/200 — mask_dice_loss: 0.056867
Epochs:  50% 99/200 [08:28<08:37,  5.12s/it]Epoch 100/200 —

In [ ]:
display(Image(filename='output/ssim/loss_curve.png', width=800))

### 5.4 NCC+SSIM

Train **NCC+SSIM** for 200 epochs on sequence 01.

Loss terms:
- Equal-weight combination of NCC and SSIM intensity losses
- `--mask-weight 1.0` — soft Dice on cell masks
- `--int-weight 0.1` — cell-restricted combined intensity loss
- `--lambda 1.0` — displacement field smoothness regularisation

In [ ]:
!python -m scripts.cell_tracking.train \
    --data-dir dataset/train \
    --sequences 01 \
    --epochs 200 --batch-size 4 --lr 1e-4 \
    --loss ncc+ssim --lambda 1.0 \
    --mask-weight 1.0 --int-weight 0.1 \
    --output-dir output/ncc_ssim \
    --save-every 50

In [ ]:
display(Image(filename='output/ncc_ssim/loss_curve.png', width=800))

### 5.5 MSE+NCC+SSIM

Train **MSE+NCC+SSIM** for 200 epochs on sequence 01.

Loss terms:
- Equal-weight combination of MSE, NCC, and SSIM intensity losses
- `--mask-weight 1.0` — soft Dice on cell masks
- `--int-weight 0.1` — cell-restricted combined intensity loss
- `--lambda 1.0` — displacement field smoothness regularisation

In [ ]:
!python -m scripts.cell_tracking.train \
    --data-dir dataset/train \
    --sequences 01 \
    --epochs 200 --batch-size 4 --lr 1e-4 \
    --loss mse+ncc+ssim --lambda 1.0 \
    --mask-weight 1.0 --int-weight 0.1 \
    --output-dir output/mse_ncc_ssim \
    --save-every 50

In [ ]:
display(Image(filename='output/mse_ncc_ssim/loss_curve.png', width=800))

## 6. Validations

Evaluate each trained model on held-out sequence 02

### 6.1 Masked MSE

Evaluate on held-out sequence 02. Includes Horn & Schunck, Farneback, and TV-L1 baselines.

In [ ]:
!python -m scripts.cell_tracking.evaluate \
    --model output/masked_mse/best.pt \
    --data-dir dataset/train \
    --gt-dir dataset/train \
    --sequences 02 \
    --output-dir output/eval_masked_mse \
    --max-pairs 0

In [ ]:
for img_path in sorted(Path('output/eval_masked_mse').glob('pair_*.png'))[:3]:
    print(img_path.name)
    display(Image(filename=str(img_path), width=900))

### 6.2 NCC

Evaluate on held-out sequence 02. Baseline numbers reused from section 6.1.

In [ ]:
!python -m scripts.cell_tracking.evaluate \
    --model output/ncc/best.pt \
    --data-dir dataset/train \
    --gt-dir dataset/train \
    --sequences 02 \
    --output-dir output/eval_ncc \
    --no-baselines \
    --max-pairs 0

In [ ]:
for img_path in sorted(Path('output/eval_ncc').glob('pair_*.png'))[:3]:
    print(img_path.name)
    display(Image(filename=str(img_path), width=900))

### 6.3 SSIM

Evaluate on held-out sequence 02. Baseline numbers reused from section 6.1.

In [ ]:
!python -m scripts.cell_tracking.evaluate \
    --model output/ssim/best.pt \
    --data-dir dataset/train \
    --gt-dir dataset/train \
    --sequences 02 \
    --output-dir output/eval_ssim \
    --no-baselines \
    --max-pairs 0

In [ ]:
for img_path in sorted(Path('output/eval_ssim').glob('pair_*.png'))[:3]:
    print(img_path.name)
    display(Image(filename=str(img_path), width=900))

### 6.4 NCC+SSIM

Evaluate on held-out sequence 02. Baseline numbers reused from section 6.1.

In [ ]:
!python -m scripts.cell_tracking.evaluate \
    --model output/ncc_ssim/best.pt \
    --data-dir dataset/train \
    --gt-dir dataset/train \
    --sequences 02 \
    --output-dir output/eval_ncc_ssim \
    --no-baselines \
    --max-pairs 0

In [ ]:
for img_path in sorted(Path('output/eval_ncc_ssim').glob('pair_*.png'))[:3]:
    print(img_path.name)
    display(Image(filename=str(img_path), width=900))

### 6.5 MSE+NCC+SSIM

Evaluate on held-out sequence 02. Baseline numbers reused from section 6.1.

In [ ]:
!python -m scripts.cell_tracking.evaluate \
    --model output/mse_ncc_ssim/best.pt \
    --data-dir dataset/train \
    --gt-dir dataset/train \
    --sequences 02 \
    --output-dir output/eval_mse_ncc_ssim \
    --no-baselines \
    --max-pairs 0

In [ ]:
for img_path in sorted(Path('output/eval_mse_ncc_ssim').glob('pair_*.png'))[:3]:
    print(img_path.name)
    display(Image(filename=str(img_path), width=900))

## 7. New Dataset Creation

Extract individual cells from every consecutive frame pair into 224x224 isolated cells (other cells zeroed out)

In [ ]:
builder = SegmentedCellBuilder(
    data_dir="dataset/train",
    output_dir="dataset/segmented_cells",
    crop_size=224,
)
builder.build(sequences=["01", "02"])

In [ ]:
crop_dir = Path("dataset/segmented_cells")
crop_files = sorted(crop_dir.glob("*.npz"))
print(f"Total segmented cell crops: {len(crop_files)}")

n_examples = 4
indices = np.linspace(0, len(crop_files) - 1, n_examples, dtype=int)

fig, axes = plt.subplots(n_examples, 4, figsize=(14, 3.5 * n_examples))

for row, idx in enumerate(indices):
    data = np.load(str(crop_files[idx]))
    name = crop_files[idx].stem

    axes[row, 0].imshow(data["source"], cmap="gray")
    axes[row, 0].set_title(f"{name}\nsource", fontsize=10)
    axes[row, 0].axis("off")

    axes[row, 1].imshow(data["target"], cmap="gray")
    axes[row, 1].set_title("target", fontsize=10)
    axes[row, 1].axis("off")

    axes[row, 2].imshow(data["source_mask"], cmap="gray")
    axes[row, 2].set_title("source mask", fontsize=10)
    axes[row, 2].axis("off")

    axes[row, 3].imshow(data["target_mask"], cmap="gray")
    axes[row, 3].set_title("target mask", fontsize=10)
    axes[row, 3].axis("off")

plt.suptitle("Per-Cell Segmented Crops — Sample Inspection", fontsize=14, y=1.0)
plt.tight_layout()
plt.show()

## 8. New Dataset Training

Train VoxelMorph with NCC+SSIM loss on the segmented cell. Sequence 01 trains, Sequence 02 validates each epoch.

In [ ]:
!python -m scripts.cell_tracking.train_segmented_cells \
    --data-dir dataset/segmented_cells \
    --epochs 200 \
    --output-dir output/segmented_cells_ncc_ssim

display(Image(filename="output/segmented_cells_ncc_ssim/loss_curve.png"))

## 9. New Dataset Validations

Evaluate Horn & Schunck, Farneback, and TV-L1 on every cell crop in the sequence-02 validation split. Writes per-baseline Dice and runtime to `output/segmented_cells_baselines/metrics.json`.

In [ ]:
!python -m scripts.cell_tracking.evaluate_segmented_cells_baselines \
    --data-dir dataset/segmented_cells \
    --val-sequence 02 \
    --output-dir output/segmented_cells_baselines

## 10. Results — Original Dataset

Comparison of classical baselines and VoxelMorph variants on the full-frame 704×544 pairs (held-out sequence 02). All Dice/MSE/runtime numbers are per-pair.

In [ ]:
def load_metrics(path: str, key: str):
    p = Path(path)
    if not p.exists():
        return None
    return json.loads(p.read_text()).get(key)


def fmt(metrics: dict) -> dict:
    dice = (
        f"{metrics['dice_mean']:.4f} ± {metrics['dice_std']:.4f}"
        if 'dice_mean' in metrics else '—'
    )
    mse = (
        f"{metrics['masked_mse_mean']*100:.4f} ± {metrics['masked_mse_std']*100:.4f}"
        if 'masked_mse_mean' in metrics else '—'
    )
    runtime = (
        f"{metrics['runtime_mean']:.4f}"
        if 'runtime_mean' in metrics else '—'
    )
    return {'Dice': dice, 'MSE (x100)': mse, 'Runtime (s)': runtime}


def find_baselines() -> dict | None:
    """Scan all output/eval_*/metrics.json and return the first that has baseline keys."""
    for eval_dir in sorted(Path('output').glob('eval_*')):
        path = eval_dir / 'metrics.json'
        if not path.exists():
            continue
        data = json.loads(path.read_text())
        if any(k in data for k in ('horn_schunck', 'farneback', 'tvl1')):
            return data
    return None


rows = []

baseline_data = find_baselines()
if baseline_data is not None:
    for key, label in [
        ('horn_schunck', 'Horn & Schunck'),
        ('farneback',    'Farneback'),
        ('tvl1',         'TV-L1'),
    ]:
        m = baseline_data.get(key)
        if m:
            rows.append({'Method': label, **fmt(m)})

for name, eval_dir in [
    ('Masked MSE',   'output/eval_masked_mse'),
    ('NCC',          'output/eval_ncc'),
    ('SSIM',         'output/eval_ssim'),
    ('NCC+SSIM',     'output/eval_ncc_ssim'),
    ('MSE+NCC+SSIM', 'output/eval_mse_ncc_ssim'),
]:
    m = load_metrics(f'{eval_dir}/metrics.json', 'vxm')
    if m:
        rows.append({'Method': name, **fmt(m)})

pd.DataFrame(rows).set_index('Method')

## 11. New Dataset Results (Baselines & NCC+SSIM)

Comparison of classical baselines and VoxelMorph (NCC+SSIM) trained on the ~2300 per-cell isolated 224×224 images. Dice / runtime computed per-crop on the held-out sequence-02 cells.

In [ ]:
def _load(path: str, key: str):
    p = Path(path)
    if not p.exists():
        return None
    return json.loads(p.read_text()).get(key)


def _fmt(m: dict) -> dict:
    dice = (
        f"{m['dice_mean']:.4f} ± {m['dice_std']:.4f}"
        if m and 'dice_mean' in m else '—'
    )
    runtime = f"{m['runtime_mean']:.4f}" if m and 'runtime_mean' in m else '—'
    return {'Dice': dice, 'Runtime (s/crop)': runtime}


rows = []

# Baselines on segmented cell crops
baseline_json = 'output/segmented_cells_baselines/metrics.json'
for key, label in [
    ('horn_schunck', 'Horn & Schunck'),
    ('farneback',    'Farneback'),
    ('tvl1',         'TV-L1'),
]:
    m = _load(baseline_json, key)
    if m:
        rows.append({'Method': label, **_fmt(m)})

# VoxelMorph trained on segmented cells
m = _load('output/segmented_cells_ncc_ssim/metrics.json', 'vxm')
if m:
    rows.append({'Method': 'VoxelMorph (NCC+SSIM)', **_fmt(m)})

pd.DataFrame(rows).set_index('Method')